# 第四阶段：Logistic、PD、PDO 与信用评分卡

本 Notebook 只调用 `src/` 中的正式实现。训练集用于筛选、交叉验证、调参与拟合；验证集用于候选方案选择；测试集标签不读取。

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling.run_stage4 import run_stage4
summary = run_stage4(PROJECT_ROOT)
summary

{'selected_scheme': '原始比例', 'selected_feature_count': 8, 'selected_features': ['RevolvingUtilizationOfUnsecuredLines', 'age', 'DebtRatio', 'severe_delinquency_flag', 'delinquency_severity_score', 'real_estate_loan_share', 'estimated_monthly_debt', 'residual_income_per_capita'], 'selected_C': 0.1, 'threshold': 0.19381917000546303, 'validation_metrics': {'auc': 0.8595863339470943, 'ks': 0.566737002590159, 'pr_auc': 0.3584065708390137, 'precision': 0.39395546129374337, 'recall': 0.4940159574468085, 'f1': 0.4383480825958702, 'brier': 0.05026725859080599, 'log_loss': 0.18160569747409874, 'calibration_intercept': 0.030214820389866406, 'calibration_slope': 1.0112508949499164}, 'grade_boundaries': [585.4187666063689, 611.0504890329896, 629.6588090911786, 640.8519940239293], 'chart_count': 8, 'test_label_read': False, 'model': 'D:\\AI project\\GMSC-Credit-Scoring\\models\\stage4_logistic_scorecard.joblib', 'report': 'D:\\AI project\\GMSC-Credit-Scoring\\reports\\excel\\stage4_logistic_scorecard

## 变量筛选轨迹

所有删除决策均来自训练集；高 IV 不代表自动入模。

In [2]:
report = PROJECT_ROOT / 'reports' / 'excel' / 'stage4_logistic_scorecard.xlsx'
selection = pd.read_excel(report, sheet_name=3)
selection

,feature,action,stage,reason
0,RevolvingUtilizationOfUnsecuredLines,保留,IV_PSI,"IV=1.110299, PSI=0.000254"
1,age,保留,IV_PSI,"IV=0.242014, PSI=0.000222"
2,NumberOfTime30-59DaysPastDueNotWorse,保留,IV_PSI,"IV=0.729666, PSI=0.000091"
3,DebtRatio,保留,IV_PSI,"IV=0.082663, PSI=0.000688"
4,MonthlyIncome,保留,IV_PSI,"IV=0.077373, PSI=0.000408"
5,NumberOfOpenCreditLinesAndLoans,保留,IV_PSI,"IV=0.083954, PSI=0.000228"
6,NumberOfTimes90DaysLate,保留,IV_PSI,"IV=0.852751, PSI=0.000146"
7,NumberRealEstateLoansOrLines,保留,IV_PSI,"IV=0.056907, PSI=0.000363"
8,NumberOfTime60-89DaysPastDueNotWorse,保留,IV_PSI,"IV=0.037471, PSI=0.000066"
9,NumberOfDependents,保留,IV_PSI,"IV=0.035388, PSI=0.000095"


## 验证集方案比较

先应用校准门槛，再综合 AUC、KS、PR-AUC、F1、Brier 与 Log Loss 选择方案。

In [3]:
comparison = pd.read_excel(report, sheet_name=7)
comparison

,auc,ks,ks_threshold,pr_auc,brier,log_loss,threshold,precision,recall,f1,calibration_intercept,calibration_slope,scheme,C,class_weight,calibration_pass,metric_wins,selected
0,0.859586,0.566737,0.051089,0.358407,0.050267,0.181606,0.193819,0.393955,0.494016,0.438348,0.030215,1.011251,原始比例,0.10,none,True,6,True
1,0.859725,0.568297,0.049642,0.352396,0.050621,0.182090,0.209802,0.393199,0.492021,0.437094,-0.166066,0.935873,类别权重,0.01,balanced,False,0,False


## PDO 计分关系

评分公式为 `Score = Offset - Factor × logit(PD)`；基础分与各变量贡献分之和必须等于未取整总分。基准分为 600，好坏 Odds 为 20:1，PDO 为 20。

In [4]:
grades = pd.read_excel(report, sheet_name=12)
grades

,split,risk_grade,count,mean_score,mean_pd,bad_rate
0,train,A,20901,647.510596,0.009638,0.005646
1,train,B,20854,635.122959,0.014669,0.010262
2,train,C,21218,621.206061,0.023809,0.023141
3,train,D,21026,598.884562,0.050823,0.057548
4,train,E,21001,553.572935,0.234937,0.237370
5,validation,A,4539,647.563654,0.009622,0.006169
6,validation,B,4467,635.140378,0.014660,0.009402
7,validation,C,4555,621.143857,0.023859,0.023052
8,validation,D,4452,599.060281,0.050545,0.057278
9,validation,E,4487,553.726737,0.234441,0.239358


## 阶段边界

本阶段指标均来自训练集或验证集。测试集保留到第五阶段完成三类模型后统一评价；本结果仅用于教学研究，不代表真实授信决策。